# Evaluación general del pipeline (Fase 3+4+5)

Evalúa el pipeline completo (modelo + heurística + confianza) sobre `evaluate/pdfs` + `evaluate/jsons`, a diferencia de `evaluate_association.ipynb`, que aísla solo la heurística.

In [1]:
import os
import sys
from pathlib import Path

ROOT = Path().resolve()
while ROOT.name != "pdf-key-extraction":
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)

In [2]:
import json
from collections import defaultdict

import pandas as pd

from extract.key_value_extractor import KeyValueExtractor

LABELED_DIR = Path("evaluate/jsons")
PDF_DIR = Path("evaluate/pdfs")

extractor = KeyValueExtractor()

C:\Users\Samuel\miniconda3\envs\pdf-key-extraction\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Etiquetas de referencia

In [3]:
def load_gold_results(json_path: Path):
    with open(json_path, encoding="utf-8") as f:
        entities = json.load(f)

    by_page = defaultdict(list)
    for e in entities:
        by_page[e["page"]].append(e)

    results = []
    for page, page_entities in sorted(by_page.items()):
        results.append({
            "page": page,
            "words": [e["text"] for e in page_entities],
            "boxes": [e["normalized_bbox"] for e in page_entities],
            "labels": [e["label"] for e in page_entities],
        })
    return results


labeled_files = sorted(LABELED_DIR.glob("*.json"))
gold_outputs = {}
for path in labeled_files:
    doc_id = path.stem
    gold_results = load_gold_results(path)
    gold_outputs[doc_id] = extractor.apply_heuristic(gold_results)

print(f"Documentos de evaluación: {len(gold_outputs)}")

Documentos de evaluación: 10


## 2. Pipeline completo

In [4]:
available_pdf_ids = {p.stem for p in PDF_DIR.glob("*.pdf")}
common_ids = sorted(available_pdf_ids & set(gold_outputs))

missing_pdf = set(gold_outputs) - available_pdf_ids
if missing_pdf:
    print(f"{len(missing_pdf)} documentos etiquetados sin PDF en evaluate/pdfs/: {missing_pdf}")

pipeline_outputs = {}
for doc_id in common_ids:
    pdf_path = PDF_DIR / f"{doc_id}.pdf"
    pipeline_outputs[doc_id] = extractor.predict(pdf_path)

C:\Users\Samuel\miniconda3\envs\pdf-key-extraction\Lib\site-packages\transformers\modeling_utils.py:1051: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(


## 3. Comparación campo a campo (form)

In [5]:
def normalize(value):
    return value.strip().upper() if isinstance(value, str) else value


def match_form(gold_form, pipeline_form):
    gold_by_field = defaultdict(list)
    for entry in gold_form:
        gold_by_field[entry["field"]].append(entry)

    pipeline_by_field = defaultdict(list)
    for entry in pipeline_form:
        pipeline_by_field[entry["field"]].append(entry)

    records = []
    fields = sorted(set(gold_by_field) | set(pipeline_by_field))
    for field in fields:
        golds = gold_by_field.get(field, [])
        preds = pipeline_by_field.get(field, [])

        for i in range(max(len(golds), len(preds))):
            gold_entry = golds[i] if i < len(golds) else None
            pred_entry = preds[i] if i < len(preds) else None

            gold_value = gold_entry["value"] if gold_entry else None
            pred_value = pred_entry["value"] if pred_entry else None
            is_reliable = pred_entry["is_reliable"] if pred_entry else None

            records.append({
                "field": field,
                "gold_value": gold_value,
                "pipeline_value": pred_value,
                "is_reliable": is_reliable,
                "correct": (normalize(gold_value) == normalize(pred_value)) if pred_entry else None,
            })
    return records


form_records = []
for doc_id in common_ids:
    for record in match_form(gold_outputs[doc_id]["form"], pipeline_outputs[doc_id]["form"]):
        record["doc_id"] = doc_id
        form_records.append(record)

form_df = pd.DataFrame(form_records)
form_df.head(10)

,field,gold_value,pipeline_value,is_reliable,correct,doc_id
0,AMBIENTE:,PRODUCCIÓN,PRODUCCIÓN,True,True,1106202601139174848500120080200000487350004873513
1,Agente de Retención Resolución No.,1,1,True,True,1106202601139174848500120080200000487350004873513
2,Contribuyente Especial,0011,0011,True,True,1106202601139174848500120080200000487350004873513
3,Direccion:,MONTECRISTI,MONTECRISTI,True,True,1106202601139174848500120080200000487350004873513
4,Dirección Matriz:,KM 3.5 VIA PORTOVIEJO-CRUCITA,KM 3.5 VIA PORTOVIEJO-CRUCITA,True,True,1106202601139174848500120080200000487350004873513
5,Dirección Sucursal:,PANAMERICANA S/N Y BOLIVAR,PANAMERICANA S/N Y BOLIVAR,True,True,1106202601139174848500120080200000487350004873513
6,EMISIÓN:,NORMAL,NORMAL,True,True,1106202601139174848500120080200000487350004873513
7,FECHA Y HORA DE AUTORIZACIÓN:,2026-06-11 16:51:06,2026-06-11 16:51:06,True,True,1106202601139174848500120080200000487350004873513
8,Fecha,2026-06-11 00:00:00,2026-06-11 00:00:00,True,True,1106202601139174848500120080200000487350004873513
9,Forma de pago,20 - OTROS CON UTILIZACION DEL SISTEMA FINANCIERO,20 - OTROS CON UTILIZACION DEL SISTEMA FINANCIERO,True,True,1106202601139174848500120080200000487350004873513


## 4. Métricas de extracción (form)

In [6]:
def precision_recall_f1(df):
    tp = int((df["correct"] == True).sum())
    fp = int(df["pipeline_value"].notna().sum()) - tp
    fn = int(df["gold_value"].notna().sum()) - tp
    precision = tp / (tp + fp) if (tp + fp) else float("nan")
    recall = tp / (tp + fn) if (tp + fn) else float("nan")
    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) else float("nan")
    )
    return {"tp": tp, "fp": fp, "fn": fn, "precision": precision, "recall": recall, "f1": f1}


form_metrics = precision_recall_f1(form_df)
form_metrics

{'tp': 263,
 'fp': 0,
 'fn': 30,
 'precision': 1.0,
 'recall': 0.8976109215017065,
 'f1': 0.9460431654676259}

## 5. Calibración de `is_reliable`

Solo es informativa si hubo 2 o más candidatos compitiendo.

In [7]:
def calibration_report(df):
    evaluated = df.dropna(subset=["correct", "is_reliable"])
    crosstab = pd.crosstab(evaluated["is_reliable"], evaluated["correct"])
    acc_reliable = evaluated.loc[evaluated["is_reliable"] == True, "correct"].mean()
    return {"crosstab": crosstab, "accuracy_if_reliable": acc_reliable}


form_calibration = calibration_report(form_df)
print(form_calibration["crosstab"])
print()
print(f"accuracy_if_reliable: {form_calibration['accuracy_if_reliable']}")

form_candidate_counts = [
    len(entry["candidates"])
    for doc_id in common_ids
    for entry in pipeline_outputs[doc_id]["form"]
]
form_multi = sum(1 for n in form_candidate_counts if n >= 2)
print(
    f"\nForm: {len(form_candidate_counts)} campos totales, "
    f"{form_multi} con >=2 candidatos ({form_multi / len(form_candidate_counts):.1%})"
)

correct      True
is_reliable      
True          263

accuracy_if_reliable: 1.0

Form: 263 campos totales, 0 con >=2 candidatos (0.0%)


## 6. Comparación de tablas

In [8]:
def match_tables(gold_tables, pipeline_tables):
    records = []
    for i in range(max(len(gold_tables), len(pipeline_tables))):
        gold_table = gold_tables[i] if i < len(gold_tables) else {"headers": [], "rows": []}
        pipeline_table = pipeline_tables[i] if i < len(pipeline_tables) else {"headers": [], "rows": []}

        headers = pipeline_table["headers"] or gold_table["headers"]
        n_rows = max(len(gold_table["rows"]), len(pipeline_table["rows"]))

        for j in range(n_rows):
            gold_row = gold_table["rows"][j] if j < len(gold_table["rows"]) else None
            pred_row = pipeline_table["rows"][j] if j < len(pipeline_table["rows"]) else None

            for col_idx, column in enumerate(headers):
                gold_cell = gold_row[col_idx] if gold_row and col_idx < len(gold_row) else None
                pred_cell = pred_row[col_idx] if pred_row and col_idx < len(pred_row) else None

                gold_value = gold_cell["value"] if gold_cell else None
                pred_value = pred_cell["value"] if pred_cell else None
                is_reliable = pred_cell["is_reliable"] if pred_cell else None

                records.append({
                    "table": i,
                    "row": j,
                    "column": column,
                    "gold_value": gold_value,
                    "pipeline_value": pred_value,
                    "is_reliable": is_reliable,
                    "correct": (normalize(gold_value) == normalize(pred_value)) if pred_cell else None,
                })
    return records


table_records = []
for doc_id in common_ids:
    for record in match_tables(gold_outputs[doc_id]["tables"], pipeline_outputs[doc_id]["tables"]):
        record["doc_id"] = doc_id
        table_records.append(record)

table_df = pd.DataFrame(table_records)
table_df.head(10)

,table,row,column,gold_value,pipeline_value,is_reliable,correct,doc_id
0,0,0,Cod. Principal,491,491,True,True,1106202601139174848500120080200000487350004873513
1,0,0,Cod. Auxiliar,None,None,None,None,1106202601139174848500120080200000487350004873513
2,0,0,Cantidad,1.0,1.0,True,True,1106202601139174848500120080200000487350004873513
3,0,0,Descripción,AVPF LUDAMA BOTOX VITRIFICx300GR,AVPF LUDAMA BOTOX VITRIFICx300GR,True,True,1106202601139174848500120080200000487350004873513
4,0,0,Detalle Adicional,None,None,None,None,1106202601139174848500120080200000487350004873513
5,0,0,Precio Unitario,13.9594,13.9594,True,True,1106202601139174848500120080200000487350004873513
6,0,0,Subsidio,0.0,0.0,True,True,1106202601139174848500120080200000487350004873513
7,0,0,Precio sin Subsidio,0.0,0.0,True,True,1106202601139174848500120080200000487350004873513
8,0,0,Descuento,0.0,0.0,True,True,1106202601139174848500120080200000487350004873513
9,0,0,Precio Total,13.96,13.96,True,True,1106202601139174848500120080200000487350004873513


## 7. Métricas de tablas

In [9]:
table_metrics = precision_recall_f1(table_df)
table_calibration = calibration_report(table_df)

print(table_metrics)
print()
print(table_calibration["crosstab"])
print()
print(f"accuracy_if_reliable: {table_calibration['accuracy_if_reliable']}")

table_candidate_counts = [
    len(cell["candidates"])
    for doc_id in common_ids
    for table in pipeline_outputs[doc_id]["tables"]
    for row in table["rows"]
    for cell in row
    if cell is not None
]
table_multi = sum(1 for n in table_candidate_counts if n >= 2)
print(
    f"\nTablas: {len(table_candidate_counts)} celdas totales, "
    f"{table_multi} con >=2 candidatos ({table_multi / len(table_candidate_counts):.1%})"
)

{'tp': 261, 'fp': 0, 'fn': 72, 'precision': 1.0, 'recall': 0.7837837837837838, 'f1': 0.8787878787878788}

correct      True
is_reliable      
True          261

accuracy_if_reliable: 1.0

Tablas: 261 celdas totales, 0 con >=2 candidatos (0.0%)


## 8. Resumen final

In [10]:
summary = pd.DataFrame([
    {"scope": "form", **form_metrics,
     "accuracy_if_reliable": form_calibration["accuracy_if_reliable"]},
    {"scope": "tables", **table_metrics,
     "accuracy_if_reliable": table_calibration["accuracy_if_reliable"]},
])
summary

,scope,tp,fp,fn,precision,recall,f1,accuracy_if_reliable
0,form,263,0,30,1.0,0.897611,0.946043,1.0
1,tables,261,0,72,1.0,0.783784,0.878788,1.0


### Conclusión

Precisión=1.0 en form y tablas; el F1 baja por el recall (30 campos y 72 celdas sin valor). Como la heurística ya se validó al 100% en 5.4, la causa es la Fase 3 (clasificación), no la asociación. `accuracy_if_reliable=1.0` pero sin ningún caso con 2+ candidatos, igual que en 5.5.